# Day 07 — Solutions: Strings, Regex, Pathlib
Runnable snippets for email extraction and kebab-case renaming.

In [ ]:
# Exercise 1 — Extract all emails (unique, lowercased)
import re
from typing import Iterable, Set

EMAIL = re.compile(r'[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}')

def extract_emails(lines: Iterable[str]) -> Set[str]:
    found: set[str] = set()
    for line in lines:
        for m in EMAIL.findall(line):
            found.add(m.lower())
    return found

text = '''
Contact alice@example.com, Bob@Example.com.
Backup: team@sub.domain.org
'''
extract_emails(text.splitlines())

In [ ]:
# Exercise 2 — Rename files to kebab-case (dry run)
from pathlib import Path
import re

_slug_chars = re.compile(r'[^a-z0-9-]+')
_multi_dash = re.compile(r'-{2,}')

def to_kebab(stem: str) -> str:
    s = stem.lower().replace('_','-').replace(' ','-')
    s = _slug_chars.sub('-', s)
    s = _multi_dash.sub('-', s).strip('-')
    return s or 'file'

def rename_dir_to_kebab(dirpath: Path, *, dry_run: bool = True) -> list[tuple[Path, Path]]:
    changes: list[tuple[Path, Path]] = []
    for p in dirpath.iterdir():
        if not p.is_file():
            continue
        new_stem = to_kebab(p.stem)
        target = p.with_name(new_stem + p.suffix.lower())
        if target == p:
            continue
        if target.exists():
            print(f'skip (exists): {target}')
            continue
        changes.append((p, target))
        if dry_run:
            print(f'DRY-RUN: {p.name} -> {target.name}')
        else:
            p.rename(target)
            print(f'renamed: {p.name} -> {target.name}')
    return changes

# Example (dry-run):
# rename_dir_to_kebab(Path('reports'), dry_run=True)